<a href="https://colab.research.google.com/github/isumakm/Weather-Prediction-and-Crop-Recommendation-System-/blob/Multi-Crop-Ranking/Rule_based_suitability_system.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##**Setup**

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score, top_k_accuracy_score
from sklearn.ensemble import RandomForestClassifier

##**Load Data**

In [ ]:
try:
    df = pd.read_csv("/content/drive/MyDrive/DSGP/Crop_training_data_FULL_final.csv")
    print("Dataset loaded correctly")
    print(df.shape)

except Exception as e:
    print("Dataset loading failed:", e)

df.head()

Dataset loaded correctly
(2100, 14)


,crop,temperature,rainfall,sunshine_hours,ph,organic_carbon,cec,awc,bulk_density,rooting_depth_m,texture,texture_code,suitability,suitability_class
0,Brinjal,17.272137,2734.849798,9.257206,7.098268,1.881237,33.767915,0.389893,0.945723,1.471861,silt loam,5,0.698,Unsuitable
1,Brinjal,23.442325,2079.442081,3.802854,7.902331,2.747803,34.757119,0.105033,1.655120,1.323896,sand,1,0.589,Unsuitable
2,Brinjal,35.281442,1567.821919,4.822195,5.368965,1.014584,6.906058,0.343163,1.050276,1.157586,loamy sand,2,0.724,Unsuitable
3,Brinjal,29.140985,985.385576,8.182605,7.006299,3.335882,23.568867,0.393432,1.408769,0.542888,loamy sand,2,0.860,Suitable
4,Brinjal,21.023598,815.960211,5.464674,7.233649,2.836555,34.949356,0.165123,1.406248,0.640387,silt loam,5,0.835,Suitable


##**Identify feature columns + label column**

In [ ]:
# ---- REQUIRED label column ----
LABEL_COL = "crop"   # your dataset uses "crop" (based on previous messages)

# ---- Candidate feature columns (we'll use what exists) ----
CANDIDATE_FEATURES = [
    # weather-like
    "temperature", "rainfall", "sunshine_hours",

    # soil-like
    "ph", "organic_carbon", "cec", "awc", "bulk_density",
    "sand_pct", "silt_pct", "clay_pct", "taw",

    # location
    "lat", "lon",

    # categorical
    "texture_class", "rooting_depth"
]

FEATURES = [c for c in CANDIDATE_FEATURES if c in df.columns]
missing = [c for c in CANDIDATE_FEATURES if c not in df.columns]

print("Using FEATURES:", FEATURES)
print("Missing columns (OK for now):", missing)

# Basic cleaning
df = df.dropna(subset=[LABEL_COL])  # label must exist
df = df.drop_duplicates()

# Keep only necessary columns
df_model = df[FEATURES + [LABEL_COL]].copy()

print(df_model.shape)
df_model.head()

Using FEATURES: ['temperature', 'rainfall', 'sunshine_hours', 'ph', 'organic_carbon', 'cec', 'awc', 'bulk_density']
Missing columns (OK for now): ['sand_pct', 'silt_pct', 'clay_pct', 'taw', 'lat', 'lon', 'texture_class', 'rooting_depth']
(2100, 9)


,temperature,rainfall,sunshine_hours,ph,organic_carbon,cec,awc,bulk_density,crop
0,17.272137,2734.849798,9.257206,7.098268,1.881237,33.767915,0.389893,0.945723,Brinjal
1,23.442325,2079.442081,3.802854,7.902331,2.747803,34.757119,0.105033,1.655120,Brinjal
2,35.281442,1567.821919,4.822195,5.368965,1.014584,6.906058,0.343163,1.050276,Brinjal
3,29.140985,985.385576,8.182605,7.006299,3.335882,23.568867,0.393432,1.408769,Brinjal
4,21.023598,815.960211,5.464674,7.233649,2.836555,34.949356,0.165123,1.406248,Brinjal


##**Preprocessing + Train/Test split**

In [ ]:
X = df_model[FEATURES]
y = df_model[LABEL_COL]

# Detect categorical vs numeric
cat_cols = [c for c in X.columns if X[c].dtype == "object"]
num_cols = [c for c in X.columns if c not in cat_cols]

print("Numeric:", num_cols)
print("Categorical:", cat_cols)

preprocess = ColumnTransformer(
    transformers=[
        ("num", "passthrough", num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

Numeric: ['temperature', 'rainfall', 'sunshine_hours', 'ph', 'organic_carbon', 'cec', 'awc', 'bulk_density']
Categorical: []


##**Train the Multi-Crop Ranking Model**

In [ ]:
rf = RandomForestClassifier(
    n_estimators=500,
    random_state=42,
    class_weight="balanced_subsample",
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    n_jobs=-1
)

model = Pipeline(steps=[
    ("prep", preprocess),
    ("rf", rf)
])

model.fit(X_train, y_train)

Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('num', 'passthrough',
                                                  ['temperature', 'rainfall',
                                                   'sunshine_hours', 'ph',
                                                   'organic_carbon', 'cec',
                                                   'awc', 'bulk_density']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  [])])),
                ('rf',
                 RandomForestClassifier(class_weight='balanced_subsample',
                                        n_estimators=500, n_jobs=-1,
                                        random_state=42))])

##**Evaluate**

In [ ]:
y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)

# Probabilities for top-k
proba = model.predict_proba(X_test)
classes = model.named_steps["rf"].classes_

top5 = top_k_accuracy_score(y_test, proba, k=min(5, len(classes)), labels=classes)

print("Accuracy:", acc)
print("Top-5 Accuracy:", top5)
print("\nClassification report:\n", classification_report(y_test, y_pred))

Accuracy: 0.04285714285714286
Top-5 Accuracy: 0.2571428571428571

Classification report:
                 precision    recall  f1-score   support

        Banana       0.04      0.05      0.04        20
  Bitter Gourd       0.00      0.00      0.00        20
       Brinjal       0.04      0.05      0.04        20
      Capsicum       0.05      0.05      0.05        20
      Cucumber       0.00      0.00      0.00        20
        Ginger       0.05      0.05      0.05        20
       Kiriala       0.11      0.10      0.11        20
         Luffa       0.00      0.00      0.00        20
    Mangosteen       0.04      0.05      0.04        20
        Manioc       0.00      0.00      0.00        20
          Okra       0.04      0.05      0.04        20
        Papaya       0.07      0.05      0.06        20
 Passion Fruit       0.14      0.15      0.15        20
     Pineapple       0.12      0.05      0.07        20
        Radish       0.06      0.05      0.06        20
      Rambuta

##**Weather aggregation helper**

In [ ]:
def aggregate_weather(weather_daily_df):
    """
    weather_daily_df: DataFrame with predicted daily weather for the growing season.
    Must contain at least: temperature OR (tmax,tmin,tmean) and rainfall/precipitation.
    Optional: sunshine_hours.
    """
    agg = {}

    # temperature handling
    if "temperature" in weather_daily_df.columns:
        agg["temperature"] = float(weather_daily_df["temperature"].mean())
    elif "tmean" in weather_daily_df.columns:
        agg["temperature"] = float(weather_daily_df["tmean"].mean())
    elif "tmax" in weather_daily_df.columns and "tmin" in weather_daily_df.columns:
        agg["temperature"] = float(((weather_daily_df["tmax"] + weather_daily_df["tmin"]) / 2).mean())

    # rainfall handling
    if "rainfall" in weather_daily_df.columns:
        agg["rainfall"] = float(weather_daily_df["rainfall"].sum())
    elif "precipitation" in weather_daily_df.columns:
        agg["rainfall"] = float(weather_daily_df["precipitation"].sum())
    elif "rain" in weather_daily_df.columns:
        agg["rainfall"] = float(weather_daily_df["rain"].sum())

    # sunshine hours (optional)
    if "sunshine_hours" in weather_daily_df.columns:
        agg["sunshine_hours"] = float(weather_daily_df["sunshine_hours"].mean())

    return agg

##**Rank crops function**

In [ ]:
def rank_top_k_crops(model, soil_output: dict, weather_agg: dict, k=5):
    """
    soil_output: dict from soil model
    weather_agg: dict from aggregate_weather()
    Returns top-k crops with probabilities
    """
    # Combine both into one feature row
    row = {}
    row.update(soil_output)
    row.update(weather_agg)

    # Make DataFrame with all expected feature columns
    X_one = pd.DataFrame([row])

    # Ensure all required training features exist (missing → NaN)
    for col in FEATURES:
        if col not in X_one.columns:
            X_one[col] = np.nan
    X_one = X_one[FEATURES]

    # Predict probabilities
    proba = model.predict_proba(X_one)[0]
    classes = model.named_steps["rf"].classes_

    ranked = sorted(zip(classes, proba), key=lambda x: x[1], reverse=True)
    return ranked[:k], ranked

##**Example run**

In [ ]:
# ---- fake soil model output ----
soil_out = {
    "lon": 80.2,
    "lat": 6.9,
    "taw": 0.22,
    "organic_carbon": 1.6,
    "cec": 18,
    "ph": 6.2,
    "sand_pct": 45,
    "texture_class": "Sandy loam"
}

# ---- fake weather daily output ----
weather_daily = pd.DataFrame({
    "temperature": np.random.normal(28, 2, 120),
    "rainfall": np.random.gamma(2, 3, 120),
    "sunshine_hours": np.random.normal(6, 1, 120)
})

weather_agg = aggregate_weather(weather_daily)

top5, full_rank = rank_top_k_crops(model, soil_out, weather_agg, k=5)

print("Weather aggregated:", weather_agg)
print("\nTop 5 crops:")
for crop, p in top5:
    print(f"{crop:20s}  probability={p:.4f}")

Weather aggregated: {'temperature': 28.14770071541708, 'rainfall': 738.9617640859142, 'sunshine_hours': 5.888702874007856}

Top 5 crops:
Banana                probability=0.1080
Kiriala               probability=0.0920
Cucumber              probability=0.0820
Yams                  probability=0.0740
Yard Long Bean        probability=0.0660
